# 📓 Day 1-2: Embeddings & Vector Indexing
## VERA (Verified Evidence Retrieval Assistant)

**Objective**: Generate dense vector embeddings from processed chunks using `BAAI/bge-base-en-v1.5` and build a persistent ChromaDB index with metadata.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.embeddings.embedder import MedicalEmbedder
from src.embeddings.vector_store import VectorStoreManager
from src.ingestion.chunker import Chunk
from src.utils.helpers import load_json

print("Imports loaded!")

Imports loaded!


### 1. Load Processed Chunks

In [2]:
chunks_data = load_json("../data/processed/chunk_catalog.json")
chunks = [Chunk(**c) for c in chunks_data]
print(f"Loaded {len(chunks)} chunks from catalog.")

Loaded 94 chunks from catalog.


### 2. Initialize Embedder & Vector Store with `BAAI/bge-base-en-v1.5`

In [3]:
embedder = MedicalEmbedder(model_name="BAAI/bge-small-en-v1.5")

vector_store = VectorStoreManager(
    persist_dir="../data/vector_db",
    collection_name="vera_clinical_guidelines",
    embedder=embedder
)

vector_store.index_chunks(chunks)


2026-08-16 21:52:07 | INFO     | src.embeddings.embedder:59 - Loading Local Model: 'BAAI/bge-small-en-v1.5' on device 'cpu'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-16 21:52:12 | SUCCESS  | src.embeddings.embedder:61 - Model 'BAAI/bge-small-en-v1.5' loaded successfully (dim=384)


d:\AI Hackathon\New data\src\embeddings\embedder.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.success(f"Model '{self.model_name}' loaded successfully (dim={self.model.get_sentence_embedding_dimension()})")


2026-08-16 21:52:13 | INFO     | src.embeddings.vector_store:44 - VectorStoreManager connected to ChromaDB collection: 'vera_clinical_guidelines' (Current count: 94)
2026-08-16 21:52:13 | INFO     | src.embeddings.vector_store:67 - Indexing 94 chunks into vector store...
2026-08-16 21:52:23 | INFO     | src.embeddings.vector_store:108 - Total records in ChromaDB: 188


188

### 3. Test Raw Vector Query

In [4]:
test_query = "What are the advantages of long-read sequencing in detecting chromosomal rearrangements?"
results = vector_store.search(test_query, top_k=5)

for i, res in enumerate(results, 1):
    print(f"\n--- Result #{i} (Similarity: {res['similarity_score']:.4f}) ---")
    print(f"Document: {res['metadata'].get('doc_name')}")
    print(f"Section: {res['metadata'].get('section')} | Page: {res['metadata'].get('page_number')}")
    print(f"Snippet: {res['content'][:200]}...")


--- Result #1 (Similarity: 0.8362) ---
Document: GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf
Section: General Overview | Page: 1
Snippet: Research A national long-read sequencing study on chromosomal rearrangements uncovers hidden complexities Jesper Eisfeldt,1,2,3,14 Adam Ameur,4,5,14 Felix Lenner,4,5 Esmee Ten Berk de Boer,1,2,3 Marle...

--- Result #2 (Similarity: 0.8362) ---
Document: GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf
Section: General Overview | Page: 1
Snippet: Research A national long-read sequencing study on chromosomal rearrangements uncovers hidden complexities Jesper Eisfeldt,1,2,3,14 Adam Ameur,4,5,14 Felix Lenner,4,5 Esmee Ten Berk de Boer,1,2,3 Marle...

--- Result #3 (Similarity: 0.7778) ---
Document: GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf
Section: General Overview | Page: 2
Snippet: Toward routine clinical long-read sequencing challenging (Cameron et al. 2019; Kosugi et al. 2019). Such Results events, collec

### 4. 🔬 Test Clinical Query on SMA Treatments

In [5]:
sma_query = "What are the approved disease-modifying therapies for Spinal Muscular Atrophy?"
sma_results = vector_store.search(sma_query, top_k=3)

for i, res in enumerate(sma_results, 1):
    print(f"\n--- SMA Result #{i} (Similarity: {res['similarity_score']:.4f}) ---")
    print(f"Document: {res['metadata'].get('doc_name')}")
    print(f"Section: {res['metadata'].get('section')} | Page: {res['metadata'].get('page_number')}")
    print(f"Snippet: {res['content'][:200]}...")


--- SMA Result #1 (Similarity: 0.7800) ---
Document: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf
Section: General Overview | Page: 1
Snippet: RESEARCHARTICLE OPENACCESS Spinal Muscular Atrophy Update in Best Practices RecommendationsforTreatmentConsiderations MaryK.Schroth,MD,JenniferDeans,MHA,MS,CCLS,DianaX.BharuchaGoebel,MD,W.BryanBurnett...

--- SMA Result #2 (Similarity: 0.7800) ---
Document: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf
Section: General Overview | Page: 1
Snippet: RESEARCHARTICLE OPENACCESS Spinal Muscular Atrophy Update in Best Practices RecommendationsforTreatmentConsiderations MaryK.Schroth,MD,JenniferDeans,MHA,MS,CCLS,DianaX.BharuchaGoebel,MD,W.BryanBurnett...

--- SMA Result #3 (Similarity: 0.7496) ---
Document: medRxiv_2024_SMA_Missed_Diagnoses_Sequencing.pdf
Section: General Overview | Page: 3
Snippet: medRxiv preprint doi: https://doi.org/10.1101/2024.02.11.24302646; this version posted February 13, 2024. The copyright holder for this preprint (